In [2]:
import sys, os
sys.path.append(os.path.abspath(".."))

from semantic_agent import SemanticAgent

agent = SemanticAgent()

## Q1 "How many loads were delivered in the last full month available in the data?"

In [42]:
q1 = "How many loads were delivered in the last full month available in the data?"
result_q1 = agent.answer_question(q1)

print(f"Question: {result_q1['question']}")
print(f"\nSQL:\n{result_q1['generated_sql']}")
print(f"\nAnswer: {result_q1['answer']}")

Question: How many loads were delivered in the last full month available in the data?

SQL:
WITH max_date AS (
  SELECT MAX(delivery_date) AS max_delivery_date
  FROM analytics.fact_loads
),
last_full_month AS (
  SELECT
    date_trunc('month', max_delivery_date) - interval '1 month' AS month_start,
    date_trunc('month', max_delivery_date) AS month_end
  FROM max_date
)
SELECT COUNT(*) AS loads_delivered
FROM analytics.fact_loads f, last_full_month lfm
WHERE f.delivery_date >= lfm.month_start
  AND f.delivery_date < lfm.month_end;

Answer: [{'loads_delivered': 0}]


## Q2 "Which shipper had the highest total book price?"

In [43]:
q2 = "Which shipper had the highest total book price?"
result_q2 = agent.answer_question(q2)

print(f"Question: {result_q2['question']}")
print(f"\nSQL:\n{result_q2['generated_sql']}")
print(f"\nAnswer: {result_q2['answer']}")

Question: Which shipper had the highest total book price?

SQL:
SELECT ds.shipper_name, SUM(fl.book_price) AS total_book_price
FROM analytics.fact_loads fl
JOIN analytics.dim_shipper ds ON fl.shipper_id = ds.shipper_id
GROUP BY ds.shipper_name
ORDER BY total_book_price DESC
LIMIT 1;

Answer: [{'shipper_name': 'Shipper 1249', 'total_book_price': 1915694.1600000034}]


## Q3 "What is the average book price per load by pickup state?"

In [40]:
q3 = "What is the average book price per load by pickup state?"
result_q3 = agent.answer_question(q3)

print(f"Question: {result_q3['question']}")
print(f"\nSQL:\n{result_q3['generated_sql']}")
print(f"\nAnswer: {result_q3['answer']}")

Question: What is the average book price per load by pickup state?

SQL:
SELECT
    dl.state AS pickup_state,
    AVG(fl.book_price) AS avg_book_price
FROM analytics.fact_loads fl
JOIN analytics.dim_lane dla ON fl.lane_id = dla.lane_id
JOIN analytics.dim_location dl ON dla.source_location_id = dl.location_id
GROUP BY dl.state
ORDER BY avg_book_price DESC;

Answer: [{'pickup_state': 'SC', 'avg_book_price': 2708.9945454545464}, {'pickup_state': 'AR', 'avg_book_price': 2632.866666666667}, {'pickup_state': 'IA', 'avg_book_price': 2469.435348837209}, {'pickup_state': 'DE', 'avg_book_price': 2242.4857142857145}, {'pickup_state': 'MS', 'avg_book_price': 2100.505}, {'pickup_state': 'OR', 'avg_book_price': 2069.6565}, {'pickup_state': 'NC', 'avg_book_price': 2009.9484716157206}, {'pickup_state': 'MO', 'avg_book_price': 1952.0044366197187}, {'pickup_state': 'WI', 'avg_book_price': 1932.873333333333}, {'pickup_state': 'ID', 'avg_book_price': 1923.7248529411766}, {'pickup_state': 'AZ', 'avg_book_p

## Q4 "What are the top 5 lanes by number of delivered loads?"

In [52]:
q4 = "What are the top 5 lanes by number of delivered loads?"
result_q4 = agent.answer_question(q4)

print(f"Question: {result_q4['question']}")
print(f"\nSQL:\n{result_q4['generated_sql']}")
print(f"\nAnswer: {result_q4['answer']}")

Question: What are the top 5 lanes by number of delivered loads?

SQL:
SELECT
    dl.lane_name,
    COUNT(*) AS delivered_load_count
FROM analytics.fact_loads fl
JOIN analytics.dim_lane dl ON fl.lane_id = dl.lane_id
WHERE fl.delivery_date IS NOT NULL
    AND fl.load_was_cancelled = FALSE
GROUP BY dl.lane_name
ORDER BY delivered_load_count DESC
LIMIT 5;

Answer: [{'lane_name': 'Hawkins,TX -> Roanoke,TX', 'delivered_load_count': 882}, {'lane_name': 'Lodi,CA -> Pacific,WA', 'delivered_load_count': 150}, {'lane_name': 'Kent,WA -> Spokane,WA', 'delivered_load_count': 94}, {'lane_name': 'Henderson,NV -> Tracy,CA', 'delivered_load_count': 87}, {'lane_name': 'Taft,CA -> Tracy,CA', 'delivered_load_count': 72}]


## Q5 "Which carrier moved the most loads into Texas?"

In [45]:
q5 = "Which carrier moved the most loads into Texas?"
result_q5 = agent.answer_question(q5)

print(f"Question: {result_q5['question']}")
print(f"\nSQL:\n{result_q5['generated_sql']}")
print(f"\nAnswer: {result_q5['answer']}")

Question: Which carrier moved the most loads into Texas?

SQL:
SELECT dc.carrier_name, COUNT(*) AS load_count
FROM analytics.fact_loads fl
JOIN analytics.dim_carrier dc ON fl.carrier_id = dc.carrier_id
JOIN analytics.dim_lane dl ON fl.lane_id = dl.lane_id
JOIN analytics.dim_location loc ON dl.target_location_id = loc.location_id
WHERE loc.state = 'TX'
GROUP BY dc.carrier_name
ORDER BY load_count DESC
LIMIT 1;

Answer: [{'carrier_name': 'Carrier 567581', 'load_count': 188}]


## Q6 "How does the average book price compare between intrastate and interstate loads?"

In [46]:
q6 = "How does the average book price compare between intrastate and interstate loads?"
result_q6 = agent.answer_question(q6)

print(f"Question: {result_q6['question']}")
print(f"\nSQL:\n{result_q6['generated_sql']}")
print(f"\nAnswer: {result_q6['answer']}")

Question: How does the average book price compare between intrastate and interstate loads?

SQL:
SELECT
    CASE
        WHEN o.state = d.state THEN 'Intrastate'
        ELSE 'Interstate'
    END AS load_type,
    COUNT(*) AS load_count,
    AVG(f.book_price) AS avg_book_price
FROM analytics.fact_loads f
JOIN analytics.dim_lane l ON f.lane_id = l.lane_id
JOIN analytics.dim_location o ON l.source_location_id = o.location_id
JOIN analytics.dim_location d ON l.target_location_id = d.location_id
GROUP BY
    CASE
        WHEN o.state = d.state THEN 'Intrastate'
        ELSE 'Interstate'
    END;

Answer: [{'load_type': 'Interstate', 'load_count': 3534, 'avg_book_price': 1709.8168902093944}, {'load_type': 'Intrastate', 'load_count': 1821, 'avg_book_price': 584.4859253157565}]


## Q7 "For the shipper with the most delivered loads, how did monthly volume change across the period covered by the data?"

In [47]:
q7 = "For the shipper with the most delivered loads, how did monthly volume change across the period covered by the data?"
result_q7 = agent.answer_question(q7)

print(f"Question: {result_q7['question']}")
print(f"\nSQL:\n{result_q7['generated_sql']}")
print(f"\nAnswer: {result_q7['answer']}")

Question: For the shipper with the most delivered loads, how did monthly volume change across the period covered by the data?

SQL:
WITH top_shipper AS (
    SELECT shipper_id
    FROM analytics.fact_loads
    WHERE delivery_date IS NOT NULL
      AND (load_was_cancelled IS FALSE OR load_was_cancelled IS NULL)
    GROUP BY shipper_id
    ORDER BY COUNT(*) DESC
    LIMIT 1
)
SELECT
    DATE_TRUNC('month', fact_loads.delivery_date) AS month,
    COUNT(*) AS delivered_loads
FROM analytics.fact_loads
JOIN top_shipper ON fact_loads.shipper_id = top_shipper.shipper_id
WHERE fact_loads.delivery_date IS NOT NULL
  AND (fact_loads.load_was_cancelled IS FALSE OR fact_loads.load_was_cancelled IS NULL)
GROUP BY DATE_TRUNC('month', fact_loads.delivery_date)
ORDER BY month;

Answer: [{'month': datetime.datetime(2024, 1, 1, 0, 0), 'delivered_loads': 127}, {'month': datetime.datetime(2024, 2, 1, 0, 0), 'delivered_loads': 108}, {'month': datetime.datetime(2024, 3, 1, 0, 0), 'delivered_loads': 178}, {'m

## Q8 "Among lanes with at least 10 delivered loads, which had the highest average book price?"

In [48]:
q8 = "Among lanes with at least 10 delivered loads, which had the highest average book price?"
result_q8 = agent.answer_question(q8)

print(f"Question: {result_q8['question']}")
print(f"\nSQL:\n{result_q8['generated_sql']}")
print(f"\nAnswer: {result_q8['answer']}")

Question: Among lanes with at least 10 delivered loads, which had the highest average book price?

SQL:
SELECT
    dl.lane_name,
    AVG(fl.book_price) AS avg_book_price,
    COUNT(*) AS delivered_load_count
FROM analytics.fact_loads fl
JOIN analytics.dim_lane dl ON fl.lane_id = dl.lane_id
WHERE fl.delivery_date IS NOT NULL
  AND (fl.load_was_cancelled IS NOT TRUE)
GROUP BY dl.lane_name
HAVING COUNT(*) >= 10
ORDER BY avg_book_price DESC
LIMIT 1;

Answer: [{'lane_name': 'Stockton,CA -> Parrish,FL', 'avg_book_price': 6800.0, 'delivered_load_count': 11}]


## Q9 (custom) "What is the overall on-time delivery rate?"

Why it matters: Measures carrier reliability and shipper satisfaction. Late deliveries damage trust and cause churn. Serves as a reputation score for carriers, influencing load allocation and pricing power. Helps Loadsmart optimize load distribution and identify underperformers.

In [49]:
q9 = "What is the overall on-time delivery rate?"
result_q9 = agent.answer_question(q9)

print(f"Question: {result_q9['question']}")
print(f"\nSQL:\n{result_q9['generated_sql']}")
print(f"\nAnswer: {result_q9['answer']}")

Question: What is the overall on-time delivery rate?

SQL:
SELECT
    AVG(CASE WHEN carrier_on_time_to_delivery THEN 1.0 ELSE 0.0 END) AS on_time_delivery_rate
FROM analytics.fact_loads;

Answer: [{'on_time_delivery_rate': Decimal('0.82558356676003734827')}]


## Q10 (custom) "What is the average price per mile by equipment type?"

Why it matters: Reveals pricing variation across equipment categories and identifies market gaps. Helps Loadsmart optimize platform economics, adjust pricing algorithms, and benchmark against competitors.

In [50]:
q10 = "What is the average price per mile by equipment type?"
result_q10 = agent.answer_question(q10)

print(f"Question: {result_q10['question']}")
print(f"\nSQL:\n{result_q10['generated_sql']}")
print(f"\nAnswer: {result_q10['answer']}")

Question: What is the average price per mile by equipment type?

SQL:
SELECT
    equipment_type,
    AVG(book_price / NULLIF(mileage, 0)) AS avg_price_per_mile
FROM analytics.fact_loads
WHERE mileage IS NOT NULL AND mileage > 0
GROUP BY equipment_type
ORDER BY avg_price_per_mile DESC;

Answer: [{'equipment_type': 'FBE', 'avg_price_per_mile': 10.086578766288653}, {'equipment_type': 'RFR', 'avg_price_per_mile': 3.775587487836579}, {'equipment_type': 'DRV', 'avg_price_per_mile': 3.344381834761303}]


## Q11 (model cannot answer) "Why was load 206690321 cancelled?"

When asked, the agent can only confirm `load_was_cancelled = True` for this
load, it cannot say why, because that information doesn't exist anywhere
in the data.

Fix: this requires a data change, the source system needs to start
send a cancellation reason 

In [51]:
q11 = "Why this load 206690321 was cancelled?"
result_q11 = agent.answer_question(q11)

print(f"Question: {result_q11['question']}")
print(f"\nSQL:\n{result_q11['generated_sql']}")
print(f"\nAnswer: {result_q11['answer']}")

Question: Why this load 206690321 was cancelled?

SQL:
SELECT
    f.loadsmart_id,
    f.load_was_cancelled,
    f.load_booked_autonomously,
    f.load_sourced_autonomously,
    f.carrier_on_time_to_pickup,
    f.carrier_on_time_to_delivery,
    f.carrier_on_time_overall,
    f.carrier_dropped_us_count,
    f.carrier_rating,
    f.book_price,
    f.source_price,
    f.pnl,
    f.mileage,
    f.equipment_type,
    f.sourcing_channel,
    f.quote_date,
    f.book_date,
    f.source_date,
    f.pickup_date,
    f.delivery_date,
    f.pickup_appointment_time,
    f.delivery_appointment_time,
    l.lane_name,
    s.shipper_name,
    c.carrier_name,
    c.vip_carrier
FROM analytics.fact_loads f
LEFT JOIN analytics.dim_lane l ON f.lane_id = l.lane_id
LEFT JOIN analytics.dim_shipper s ON f.shipper_id = s.shipper_id
LEFT JOIN analytics.dim_carrier c ON f.carrier_id = c.carrier_id
WHERE f.loadsmart_id = 206690321;

Answer: [{'loadsmart_id': 206690321, 'load_was_cancelled': True, 'load_booked_auto

## Q12 (bonus) "What is the average delivery time?"

This one has no single agreed definition in the data or docs. Two equally
valid readings give very different numbers:
- `delivery_date - book_date` (from booking to completion, includes dwell
  time waiting for pickup): **158.9 hours**
- `delivery_date - pickup_date` (actual time on the road): **34.2 hours**
  (~4.6x less)

Not a data quality issue — `pickup_date`, `book_date`, and `delivery_date`
are all populated and correct. What's missing is a documented definition.

Assumption: "delivery time" = `delivery_date - book_date` — the full time
Loadsmart is on the hook for the load, from booking to completion, which is
what a shipper actually experiences waiting for their freight.

Fix: add a note to `book_date`/`delivery_date` in schema.yml recording this
assumption, and clarifying that `delivery_date - pickup_date` is a different
metric (pure on-the-road transit time) that shouldn't be confused with it.

In [4]:
q12 = "What is the average time that the loads take?"
result_q12 = agent.answer_question(q12)

print(f"Question: {result_q12['question']}")
print(f"\nSQL:\n{result_q12['generated_sql']}")
print(f"\nAnswer: {result_q12['answer']}")

Question: What is the average time that the loads take?

SQL:
SELECT AVG(delivery_date - pickup_date) AS average_load_time
FROM analytics.fact_loads
WHERE pickup_date IS NOT NULL AND delivery_date IS NOT NULL;

Answer: [{'average_load_time': datetime.timedelta(days=1, seconds=285, microseconds=53221)}]
